# Precios de Acciones — Entrenando un Agente
**Anáhuac Mayab** | Aprendizaje por Refuerzo

Referencia: Karim, R. (2018). *Tensorflow: Powerful predictive analytics with tensorflow*. Packt Publishing.

## Celda 1 — Instalación de librerías

In [ ]:
# Instalar dependencias (ejecutar solo una vez)
!pip install yfinance numpy matplotlib tensorflow

## Celda 2 — Importaciones

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf
import os
import random
import tensorflow as tf

print("TensorFlow version:", tf.__version__)
print("Librerías cargadas correctamente ✓")

## Celda 3 — Prueba rápida con yfinance
Equivalente a la prueba del Shell del PDF, pero usando `yfinance` (reemplazo moderno de `yahoo_finance`).

In [ ]:
# Prueba rápida — precio actual de Microsoft
msft = yf.Ticker('MSFT')
hist_hoy = msft.history(period='2d')

precio_apertura = hist_hoy['Open'].iloc[0]
precio_cierre   = hist_hoy['Close'].iloc[-1]
fecha           = hist_hoy.index[-1]

print(f"Símbolo  : MSFT")
print(f"Apertura : ${precio_apertura:.2f}")
print(f"Cierre   : ${precio_cierre:.2f}")
print(f"Fecha    : {fecha.date()}")

# Interpretación
cambio = ((precio_cierre - precio_apertura) / precio_apertura) * 100
print(f"\nCambio del día: {cambio:+.2f}%")

## Celda 4 — Función `get_prices()`: obtener precios históricos
Descarga los precios de apertura y los guarda en caché para evitar descargas repetidas.

In [ ]:
def get_prices(share_symbol, start_date, end_date, cache_filename):
    """
    Descarga (o carga desde caché) los precios de apertura de una acción.
    
    Parámetros:
        share_symbol   : símbolo de la acción (ej. 'MSFT')
        start_date     : fecha de inicio 'YYYY-MM-DD'
        end_date       : fecha de fin    'YYYY-MM-DD'
        cache_filename : archivo .npy para guardar/cargar los datos
    
    Retorna:
        numpy array con precios de apertura
    """
    try:
        # Intentar cargar desde caché
        stock_prices = np.load(cache_filename)
        print(f"Datos cargados desde caché: {cache_filename}")
    except IOError:
        # Descargar desde Yahoo Finance
        print(f"Descargando datos de {share_symbol}...")
        ticker     = yf.Ticker(share_symbol)
        stock_hist = ticker.history(start=start_date, end=end_date)
        stock_prices = stock_hist['Open'].values
        np.save(cache_filename, stock_prices)
        print(f"Datos guardados en: {cache_filename}")
    
    return stock_prices

## Celda 5 — Función `plot_prices()`: graficar precios

In [ ]:
def plot_prices(prices, title='Opening Stock Prices', save_as='prices.png'):
    """
    Grafica los precios de apertura de una acción.
    """
    plt.figure(figsize=(12, 5))
    plt.title(title)
    plt.xlabel('Día')
    plt.ylabel('Precio ($)')
    plt.plot(prices, color='steelblue', linewidth=0.8)
    plt.tight_layout()
    plt.savefig(save_as)
    plt.show()
    print(f"Gráfica guardada como: {save_as}")

## Celda 6 — Obtener y graficar datos históricos de MSFT
> **Nota del PDF:** Se usa un rango amplio de fechas para obtener mejor perspectiva. Aquí se actualiza al rango actual.

In [ ]:
# Rango actualizado: últimos ~7 años hasta la fecha actual
prices = get_prices(
    share_symbol   = 'MSFT',
    start_date     = '2017-01-01',
    end_date       = '2025-01-01',
    cache_filename = 'historical_stock_prices.npy'
)

print(f"\nTotal de días de datos: {len(prices)}")
print(f"Precio mínimo: ${prices.min():.2f}")
print(f"Precio máximo: ${prices.max():.2f}")

plot_prices(prices, title='Precios de apertura MSFT (2017–2025)')

## Celda 7 — Clase base `DecisionPolicy`
Clase abstracta que define la interfaz para cualquier política de decisión.

In [ ]:
class DecisionPolicy:
    """
    Clase abstracta para políticas de decisión de trading.
    
    Métodos que deben implementarse:
        select_action(current_state, step) -> acción ('Buy', 'Sell', 'Hold')
        update_q(state, action, reward, next_state) -> actualiza la función Q
    """
    def select_action(self, current_state, step):
        pass

    def update_q(self, state, action, reward, next_state):
        pass

print("Clase DecisionPolicy definida ✓")

## Celda 8 — `RandomDecisionPolicy`: política aleatoria
Hereda de `DecisionPolicy`. Selecciona acciones al azar sin considerar el estado — sirve como línea base.

In [ ]:
class RandomDecisionPolicy(DecisionPolicy):
    """
    Política de decisión aleatoria (baseline).
    Selecciona Buy, Sell o Hold al azar en cada paso.
    """
    def __init__(self, actions):
        self.actions = actions

    def select_action(self, current_state, step):
        # Ignoramos el estado — elegimos al azar
        action = self.actions[random.randint(0, len(self.actions) - 1)]
        return action

print("Clase RandomDecisionPolicy definida ✓")

## Celda 9 — Función `run_simulation()`: simulación de trading
Itera sobre los precios históricos usando una ventana deslizante (sliding window) de tamaño `hist`.

**Fórmulas clave:**
```
portfolio = budget + num_stocks * share_value
reward    = new_portfolio - current_portfolio
```

In [ ]:
def run_simulation(policy, initial_budget, initial_num_stocks, prices, hist, debug=False):
    """
    Simula el trading con la política dada sobre los precios históricos.
    
    Parámetros:
        policy             : objeto que hereda de DecisionPolicy
        initial_budget     : presupuesto inicial en dólares
        initial_num_stocks : número de acciones iniciales
        prices             : array de precios históricos
        hist               : tamaño del historial (ventana deslizante)
        debug              : imprimir estado final
    
    Retorna:
        Valor final del portafolio
    """
    budget      = initial_budget
    num_stocks  = initial_num_stocks
    share_value = 0
    transitions = []

    # El estado es: [últimos 'hist' precios, budget, num_stocks] → 202 dimensiones
    for i in range(len(prices) - hist - 1):

        # Mostrar progreso cada 100 pasos
        if i % 100 == 0:
            pct = float(100 * i) / (len(prices) - hist - 1)
            print(f'  Progreso: {pct:.2f}%', end='\r')

        # Estado actual: historial de precios + budget + num_stocks
        current_state = np.asmatrix(
            np.hstack((prices[i:i + hist], budget, num_stocks))
        )

        # Portafolio actual
        current_portfolio = budget + num_stocks * share_value

        # Seleccionar acción según la política
        action = policy.select_action(current_state, i)

        # Precio actual de la acción (siguiente en la ventana)
        share_value = float(prices[i + hist + 1])

        # Ejecutar acción
        if action == 'Buy' and budget >= share_value:
            budget     -= share_value
            num_stocks += 1
        elif action == 'Sell' and num_stocks > 0:
            budget     += share_value
            num_stocks -= 1
        else:
            action = 'Hold'  # Si no se puede comprar/vender → Hold

        # Nuevo portafolio y recompensa
        new_portfolio = budget + num_stocks * share_value
        reward        = new_portfolio - current_portfolio

        # Siguiente estado
        next_state = np.asmatrix(
            np.hstack((prices[i + 1:i + hist + 1], budget, num_stocks))
        )

        # Guardar transición y actualizar política
        transitions.append((current_state, action, reward, next_state))
        policy.update_q(current_state, action, reward, next_state)

    # Valor final del portafolio
    portfolio = budget + num_stocks * share_value

    if debug:
        print(f'\n  Presupuesto final : ${budget:.2f}')
        print(f'  Acciones en mano  : {num_stocks}')
        print(f'  Portafolio final  : ${portfolio:.2f}')

    return portfolio

print("Función run_simulation definida ✓")

## Celda 10 — Función `run_simulations()`: promedio de múltiples ejecuciones
Ejecuta la simulación 100 veces para obtener un promedio más confiable.

In [ ]:
def run_simulations(policy, budget, num_stocks, prices, hist, num_tries=100):
    """
    Corre la simulación múltiples veces y retorna promedio y desviación estándar.
    """
    final_portfolios = []

    for i in range(num_tries):
        print(f"Simulación {i + 1}/{num_tries}", end='\r')
        final_portfolio = run_simulation(policy, budget, num_stocks, prices, hist)
        final_portfolios.append(final_portfolio)

    avg = np.mean(final_portfolios)
    std = np.std(final_portfolios)
    print(f"\n\nResultados ({num_tries} simulaciones):")
    print(f"  Portafolio promedio : ${avg:.2f}")
    print(f"  Desviación estándar : ${std:.2f}")
    print(f"  Ganancia estimada   : ${avg - budget:.2f}")
    return avg, std

print("Función run_simulations definida ✓")

## Celda 11 — Evaluación con política aleatoria
3 acciones: `Buy`, `Sell`, `Hold` | historial = 200 | presupuesto = $1,000 | 100 simulaciones

In [ ]:
# Parámetros del agente
actions    = ['Buy', 'Sell', 'Hold']
hist       = 200
budget     = 1000.0
num_stocks = 0

# Política aleatoria
random_policy = RandomDecisionPolicy(actions)

# Ejecutar simulaciones
print("=== Política Aleatoria ===")
avg, std = run_simulations(random_policy, budget, num_stocks, prices, hist, num_tries=10)

# Nota: se usan 10 simulaciones por velocidad; el PDF usa 100

## Celda 12 — `QLearningDecisionPolicy`: política con Red Neuronal (Q-Learning)

Arquitectura (ver Figura 6 del PDF):
- **Entrada**: vector de estado de 202 dimensiones (200 precios + budget + num_stocks)
- **Capas ocultas**: 2 capas densas con activación ReLU
- **Salida**: 3 valores Q — uno por acción (Buy, Sell, Hold)

El hiperparámetro **epsilon** controla la exploración: a menor epsilon, más exploración aleatoria.

In [ ]:
class QLearningDecisionPolicy(DecisionPolicy):
    """
    Política Q-Learning usando una red neuronal con TensorFlow 2.
    
    - select_action : explota la mejor acción con probabilidad epsilon
    - update_q      : actualiza los pesos de la red con la experiencia (s, a, r, s')
    """

    def __init__(self, actions, input_dim):
        # ── Hiperparámetros ──────────────────────────────────────────
        self.epsilon = 0.9    # Probabilidad de explotar (vs explorar)
        self.gamma   = 0.001  # Factor de descuento
        self.actions = actions

        output_dim = len(actions)  # 3 salidas: Buy, Sell, Hold
        h1_dim     = 200           # Nodos en capa oculta

        # ── Red neuronal con Keras ───────────────────────────────────
        self.model = tf.keras.Sequential([
            tf.keras.layers.Dense(h1_dim,     activation='relu',
                                  input_shape=(input_dim,)),
            tf.keras.layers.Dense(output_dim, activation='relu')
        ])

        self.model.compile(
            optimizer=tf.keras.optimizers.SGD(learning_rate=0.01),
            loss='mse'
        )

    # ────────────────────────────────────────────────────────────────
    def select_action(self, current_state, step):
        """
        Epsilon-greedy:
          - Con probabilidad < threshold → explorar (acción aleatoria)
          - Con probabilidad ≥ threshold → explotar (mejor Q)
        """
        threshold = min(self.epsilon, step / 1000.)

        if random.random() < threshold:
            # Explotar: elegir la acción con mayor valor Q
            state_input  = np.array(current_state).reshape(1, -1)
            action_q_vals = self.model.predict(state_input, verbose=0)
            action_idx   = np.argmax(action_q_vals)
            action       = self.actions[action_idx]
        else:
            # Explorar: acción aleatoria
            action = self.actions[random.randint(0, len(self.actions) - 1)]

        return action

    # ────────────────────────────────────────────────────────────────
    def update_q(self, state, action, reward, next_state):
        """
        Actualiza la función Q usando la ecuación de Bellman:
            Q(s,a) ← reward + gamma * max_a' Q(s', a')
        """
        state_input      = np.array(state).reshape(1, -1)
        next_state_input = np.array(next_state).reshape(1, -1)

        # Q-values actuales
        action_q_vals      = self.model.predict(state_input,      verbose=0)
        next_action_q_vals = self.model.predict(next_state_input, verbose=0)

        # Índice de la acción tomada
        action_idx = self.actions.index(action)

        # Mejor Q del siguiente estado
        next_action_idx = np.argmax(next_action_q_vals)

        # Actualizar Q(s, a) con ecuación de Bellman
        action_q_vals[0, action_idx] = (
            reward + self.gamma * next_action_q_vals[0, next_action_idx]
        )

        # Entrenar la red con el nuevo target
        self.model.fit(
            state_input,
            action_q_vals,
            epochs=1,
            verbose=0
        )

print("Clase QLearningDecisionPolicy definida ✓")

## Celda 13 — Evaluación con política Q-Learning
El estado tiene **202 dimensiones** (200 precios + budget + num_stocks).

In [ ]:
# Dimensión del vector de estado: hist + 2
input_dim  = hist + 2  # 202

# Crear política Q-Learning
ql_policy = QLearningDecisionPolicy(actions, input_dim)
ql_policy.model.summary()

print("\n=== Q-Learning (una simulación con debug) ===")
portfolio_final = run_simulation(
    ql_policy, budget, num_stocks, prices, hist, debug=True
)
print(f"\nPortafolio final: ${portfolio_final:.2f}")
print(f"Ganancia/Pérdida: ${portfolio_final - budget:.2f}")

## Celda 14 — Comparación de resultados
Visualización final comparando política aleatoria vs Q-Learning.

In [ ]:
resultados = {
    'Política Aleatoria': avg,
    'Q-Learning': portfolio_final
}

fig, ax = plt.subplots(figsize=(8, 5))
colores  = ['#E07B39', '#2E86AB']
barras   = ax.bar(resultados.keys(), resultados.values(), color=colores, width=0.4)

# Línea de presupuesto inicial
ax.axhline(y=budget, color='gray', linestyle='--', label=f'Presupuesto inicial (${budget:.0f})')

# Etiquetas en barras
for barra in barras:
    height = barra.get_height()
    ax.text(barra.get_x() + barra.get_width() / 2., height + 10,
            f'${height:.2f}', ha='center', va='bottom', fontweight='bold')

ax.set_ylabel('Valor del Portafolio ($)')
ax.set_title('Comparación de Políticas — Agente de Trading')
ax.legend()
plt.tight_layout()
plt.savefig('comparacion_politicas.png', dpi=150)
plt.show()
print("Gráfica guardada como: comparacion_politicas.png")

---
## Resumen del flujo del agente

```
Infer(s)  → a         # Seleccionar acción dado el estado
Do(s, a)  → r, s'     # Ejecutar acción, obtener recompensa y nuevo estado  
Learn(s, r, a, s')    # Actualizar función Q con la nueva experiencia
```

| Componente | Descripción |
|---|---|
| **Estado (s)** | Vector 202-dim: 200 precios + budget + num_stocks |
| **Acciones** | Buy, Sell, Hold |
| **Recompensa** | `new_portfolio - current_portfolio` |
| **Política** | Red neuronal con Q-Learning (epsilon-greedy) |
| **Datos** | Precios históricos MSFT vía `yfinance` |